# Phase 1: Data Understanding



In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load All Datasets
We will identify all CSV files in the `data/raw/` directory and load them into a dictionary of DataFrames.

In [ ]:
data_path = '../data/raw/'
csv_files = glob.glob(os.path.join(data_path, '*.csv'))

datasets = {}
for file in csv_files:
    name = os.path.basename(file).split('.')[0]
    datasets[name] = pd.read_csv(file)
    print(f"Loaded {name} dataset.")

## 2. Dataset Overview
Display the shape, column names, data types, missing values, and duplicate rows for each dataset.

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name.upper()}")
    print(f"{'='*50}")
    
    # Shape
    print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Duplicates
    duplicates = df.duplicated().sum()
    print(f"Duplicate Rows: {duplicates}")
    
    # Column info summary (Data types and Missing values)
    info_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        '% Missing': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print("\nColumn Information:")
    display(info_df)

## 3. Descriptive Statistics
Generate descriptive statistics for numerical columns in each dataset.

In [ ]:
for name, df in datasets.items():
    print(f"\nDataset: {name.upper()} - Descriptive Statistics")
    display(df.describe())

## 4. Column Types (Categorical, Numerical, Datetime)
Identify which columns fall into categorical, numerical, and date/time categories.

In [ ]:
for name, df in datasets.items():
    print(f"\nDataset: {name.upper()}")
    
    # Heuristic for datetime: column name contains 'date' or 'time', or pandas can infer it easily
    date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Remove date columns from categorical if they were parsed as objects
    cat_cols = [c for c in cat_cols if c not in date_cols]
    
    print(f"- Numerical Columns: {num_cols}")
    print(f"- Categorical Columns: {cat_cols}")
    print(f"- Date/Time Columns: {date_cols}")

## 5. Unique Cities, Time Coverage, and Dataset Frequency
For datasets containing city and date information, determine the unique cities, time coverage, and recording frequency.

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name.upper()}")
    
    # Check for City
    city_col = next((col for col in df.columns if 'city' in col.lower()), None)
    if city_col:
        cities = df[city_col].nunique()
        print(f"Unique Cities: {cities}")
        
    # Check for Date
    date_col = next((col for col in df.columns if 'date' in col.lower()), None)
    if date_col:
        # Parsing may take a moment for large datasets
        temp_date = pd.to_datetime(df[date_col], errors='coerce')
        min_date = temp_date.min()
        max_date = temp_date.max()
        print(f"Time Coverage: {min_date.date()} to {max_date.date()}")
        
        # Check frequency (daily or hourly)
        if 'hour' in name.lower():
            print("Dataset Frequency: Hourly")
        elif 'day' in name.lower():
            print("Dataset Frequency: Daily")
        else:
            print("Dataset Frequency: Unspecified/Variable")


## Initial Observations

Based on the code above, executing it will reveal several facts about this dataset:

1. **Missing Values**: There is typically a large percentage of missing values across pollutant columns (PM2.5, PM10, etc.) in both city and station datasets. This necessitates careful imputation in the next phase.
2. **Data Granularity**: The dataset is provided at hourly and daily frequencies, and grouped by city or station. For baseline modeling, `city_day.csv` is usually preferred.
3. **Data Types**: Dates load as object strings initially and must be converted to datetimes during preprocessing.
4. **Categorical Features**: Variables like `AQI_Bucket` represent air quality categories and serve as classification targets or categorical features.
5. **Anomalies**: Descriptive statistics will likely show 0s or extreme maximum values for certain pollutants, indicating potential outliers that need handling.